In [2]:
import pyspark
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder.master("local[*]").appName('test').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/10 16:17:56 WARN Utils: Your hostname, Nikolas-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.16 instead (on interface en0)
26/11/10 16:17:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/10 16:18:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/10 16:18:27 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/11/10 16:18:27 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [4]:
df = spark.read.option("header", "true").csv('data/raw/green/2021/01/')

In [9]:
df.printSchema()

root
 |-- VendorID: string (nullable = true)
 |-- lpep_pickup_datetime: string (nullable = true)
 |-- lpep_dropoff_datetime: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- RatecodeID: string (nullable = true)
 |-- PULocationID: string (nullable = true)
 |-- DOLocationID: string (nullable = true)
 |-- passenger_count: string (nullable = true)
 |-- trip_distance: string (nullable = true)
 |-- fare_amount: string (nullable = true)
 |-- extra: string (nullable = true)
 |-- mta_tax: string (nullable = true)
 |-- tip_amount: string (nullable = true)
 |-- tolls_amount: string (nullable = true)
 |-- ehail_fee: string (nullable = true)
 |-- improvement_surcharge: string (nullable = true)
 |-- total_amount: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- trip_type: string (nullable = true)
 |-- congestion_surcharge: string (nullable = true)



In [8]:
import pandas as pd
import os

In [10]:
df_green_pd = pd.read_csv('data/raw/green/2021/01/green_tripdata_2021_01.csv.gz', nrows=1000)

In [11]:
from pyspark.sql import types

In [12]:
green_schema = types.StructType([
    types.StructField("VendorID", types.IntegerType(), True), 
    types.StructField("lpep_pickup_datetime", types.TimestampType(), True), 
    types.StructField("lpep_dropoff_datetime", types.TimestampType(), True), 
    types.StructField("store_and_fwd_flag", types.StringType(), True), 
    types.StructField("RatecodeID", types.IntegerType(), True), 
    types.StructField("PULocationID", types.IntegerType(), True), 
    types.StructField("DOLocationID", types.IntegerType(), True), 
    types.StructField("passenger_count", types.IntegerType(), True), 
    types.StructField("trip_distance", types.DoubleType(), True), 
    types.StructField("fare_amount", types.DoubleType(), True), 
    types.StructField("extra", types.DoubleType(), True), 
    types.StructField("mta_tax", types.DoubleType(), True), 
    types.StructField("tip_amount", types.DoubleType(), True), 
    types.StructField("tolls_amount", types.DoubleType(), True), 
    types.StructField("ehail_fee", types.DoubleType(), True), 
    types.StructField("improvement_surcharge", types.DoubleType(), True), 
    types.StructField("total_amount", types.DoubleType(), True), 
    types.StructField("payment_type", types.IntegerType(), True), 
    types.StructField("trip_type", types.IntegerType(), True), 
    types.StructField("congestion_surcharge", types.DoubleType(), True)
])

In [14]:
year = 2021

for month in range(1, 13):
    input_path = f'data/raw/green/{year}/{month:02d}/'
    output_path = f'data/pq/green/{year}/{month:02d}/'

    if not os.path.exists(input_path):
        print(f'skipping {year}/{month:02d}, no data')
        continue

    if os.path.exists(os.path.join(output_path, '_SUCCESS')):
        print(f'skipping {year}/{month:02d}, already processed')
        continue

    print(f'processing data for {year}/{month:02d}')

    df_green = spark.read \
        .option("header", "true") \
        .schema(green_schema) \
        .csv(input_path)

    df_green \
        .repartition(4) \
        .write.mode('overwrite').parquet(output_path)

processing data for 2021/01
processing data for 2021/02
processing data for 2021/03
processing data for 2021/04
processing data for 2021/05
processing data for 2021/06
processing data for 2021/07
skipping 2021/08, no data
skipping 2021/09, no data
skipping 2021/10, no data
skipping 2021/11, no data
skipping 2021/12, no data


In [15]:
df_yellow_pd = pd.read_csv('data/raw/yellow/2021/01/yellow_tripdata_2021_01.csv.gz', nrows=1000)

In [16]:
spark.createDataFrame(df_yellow_pd).schema

StructType([StructField('VendorID', LongType(), True), StructField('tpep_pickup_datetime', StringType(), True), StructField('tpep_dropoff_datetime', StringType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True)])

In [17]:
yellow_schema = types.StructType([
    types.StructField('VendorID', types.IntegerType(), True), 
    types.StructField('tpep_pickup_datetime', types.TimestampType(), True), 
    types.StructField('tpep_dropoff_datetime', types.TimestampType(), True), 
    types.StructField('passenger_count', types.IntegerType(), True), 
    types.StructField('trip_distance', types.DoubleType(), True), 
    types.StructField('RatecodeID', types.IntegerType(), True), 
    types.StructField('store_and_fwd_flag', types.StringType(), True), 
    types.StructField('PULocationID', types.IntegerType(), True), 
    types.StructField('DOLocationID', types.IntegerType(), True), 
    types.StructField('payment_type', types.IntegerType(), True), 
    types.StructField('fare_amount', types.DoubleType(), True), 
    types.StructField('extra', types.DoubleType(), True), 
    types.StructField('mta_tax', types.DoubleType(), True), 
    types.StructField('tip_amount', types.DoubleType(), True), 
    types.StructField('tolls_amount', types.DoubleType(), True), 
    types.StructField('improvement_surcharge', types.DoubleType(), True), 
    types.StructField('total_amount', types.DoubleType(), True), 
    types.StructField('congestion_surcharge', types.DoubleType(), True)
])

In [19]:
df_yellow = spark.read \
    .option("header", "true") \
    .schema(yellow_schema) \
    .csv('data/raw/yellow/2021/01/')

In [21]:
year = 2020

for month in range(1, 13):
    input_path = f'data/raw/yellow/{year}/{month:02d}/'
    output_path = f'data/pq/yellow/{year}/{month:02d}/'

    if not os.path.exists(input_path):
        print(f'skipping {year}/{month:02d}, no data')
        continue

    if os.path.exists(os.path.join(output_path, '_SUCCESS')):
        print(f'skipping {year}/{month:02d}, already processed')
        continue

    print(f'processing data for {year}/{month:02d}')

    df_yellow = spark.read \
        .option("header", "true") \
        .schema(yellow_schema) \
        .csv(input_path)

    df_yellow \
        .repartition(4) \
        .write.mode('overwrite').parquet(output_path)

processing data for 2020/01


processing data for 2020/02


processing data for 2020/03


processing data for 2020/04
processing data for 2020/05


processing data for 2020/06


processing data for 2020/07


processing data for 2020/08


processing data for 2020/09


processing data for 2020/10


processing data for 2020/11


processing data for 2020/12
